# L3 — Delta Lake: history, deletes and time travel

A Delta table records every change as a new **version** in `_delta_log/`. This notebook makes a
few changes to a small table and then looks back at them: the commit history, what a delete does
to the files on disk, reading old versions (time travel) and rolling back.

The folder layout and log files are explained in [delta.md](delta.md).

**Setup**, from the repo root:

```sh
uv sync
uv run python labs/L3/avro_write_synthetic_data.py   # creates data/users_big.avro
uv run jupyter lab                                   # then open labs/L3/delta_history.ipynb
```

The notebook works on its **own table, `data/users_history.delta`, which is deleted and rebuilt
at the start of every run**, so the version numbers below always match the text.
`data/users_big.delta` from `delta_example.py` is not touched.

Run the cells in order.

In [ ]:
import os
import shutil
from pathlib import Path


def find_repo_root() -> Path:
    """The directory holding get_data.py, searching upward from the working directory.

    A kernel starts in the notebook's own directory (labs/L3/), so walk up rather than
    assuming anything about where you launched Jupyter from.
    """
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "get_data.py").exists():
            return candidate
    raise RuntimeError("Could not find the repo root — open this notebook from inside the repo.")


# Data paths below are relative to the repo root
os.chdir(find_repo_root())
print(f"working directory: {Path.cwd()}")

In [ ]:
import pandas as pd
import pyarrow.parquet as pq
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

builder = (
    SparkSession.builder.appName("L3-delta-history")
    .master("local[4]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # Small splits, as in spark_avro_read_parquet_write.py: users_big.avro -> 3 partitions -> 3 data files
    .config("spark.sql.files.maxPartitionBytes", "100000")
    .config("spark.log.level", "ERROR")
    .config("spark.ui.showConsoleProgress", "false")  # no [Stage ...] progress bars in cell outputs
)
# configure_spark_with_delta_pip *replaces* spark.jars.packages, so spark-avro goes in extra_packages
spark = configure_spark_with_delta_pip(builder, extra_packages=["org.apache.spark:spark-avro_2.13:4.2.0"]).getOrCreate()

print(f"Spark {spark.version}, UI: {spark.sparkContext.uiWebUrl}")

In [ ]:
from pyspark_demo.display_utils import display_dataframe

SOURCE = "data/users_big.avro"
TABLE = Path("data/users_history.delta").resolve()


def show_history(delta_table: DeltaTable) -> None:
    """The table's commits, newest first, without the columns that are always empty locally."""
    columns = ["version", "timestamp", "operation", "operationParameters", "operationMetrics"]
    display_dataframe(delta_table.history().select(columns).toPandas())


def show_files(delta_table: DeltaTable) -> None:
    """Every Parquet file in the table folder, oldest first, and whether the current version uses it."""
    # A fresh read: DeltaTable.toDF().inputFiles() keeps listing the files from when forPath() was called
    current = {Path(uri).name for uri in spark.read.format("delta").load(str(TABLE)).inputFiles()}
    files = pd.DataFrame(
        [
            {
                "file": path.name[:19] + "…",
                "rows": pq.ParquetFile(path).metadata.num_rows,
                "bytes": path.stat().st_size,
                "in current version": path.name in current,
            }
            for path in sorted(TABLE.glob("*.parquet"), key=lambda p: p.stat().st_mtime)
        ]
    )
    display_dataframe(files)

## 1. Create the table — version 0

Read the Avro file and write it as a Delta table. The first commit is **version 0**: 20,000 rows
in 3 Parquet files.

In [ ]:
# Start from scratch on every run, so the version numbers match the text
shutil.rmtree(TABLE, ignore_errors=True)

spark.read.format("avro").load(SOURCE).write.format("delta").save(str(TABLE))

users = DeltaTable.forPath(spark, str(TABLE))
print("rows:", users.toDF().count())
show_history(users)
show_files(users)

## 2. Inside `_delta_log`

**A Delta table is a folder of ordinary Parquet files plus a transaction log (`_delta_log/`) that
says which of those files make up the table.** The data files are plain Parquet: pyarrow, DuckDB or
`spark.read.parquet` can open them. Reading the *folder* as plain Parquet is wrong, though, because
it also picks up files the log has already removed.

Every commit is one file, named after the table **version**, zero-padded to 20 digits:
`00000000000000000003.json` is version 3. Next to each one Delta writes `NNN.crc`, its own summary
of the table at that version, and the local filesystem adds hidden Hadoop checksums
(`.NNN.json.crc`), the same sidecars [parquet.md](parquet.md) describes.

In [ ]:
def show_log() -> None:
    """Everything in the table folder: data files at the top, then the log."""
    for path in sorted(TABLE.iterdir()):
        if path.is_file():
            print(f"  {path.name:<48} {path.stat().st_size:>8} B")
    print("  _delta_log/")
    for path in sorted((TABLE / "_delta_log").iterdir()):
        size = "dir" if path.is_dir() else f"{path.stat().st_size:>8} B"
        print(f"      {path.name:<44} {size}")


show_log()

Each commit file holds **one JSON action per line**:

| Action | Meaning |
|---|---|
| `commitInfo` | Audit record: when, which operation, its parameters and metrics. This is what `history()` shows. |
| `metaData` | Table id, format, **schema**, partition columns, table properties. Written on the first commit and again whenever they change. |
| `protocol` | Minimum reader/writer versions a client needs. Enabling features like deletion vectors raises them. |
| `add` | "This file is now part of the table": path, size, and **stats** (row count, min/max per column, null counts). |
| `remove` | "This file is no longer part of the table" — written by the delete in section 4. |

**Why the stats matter.** For `WHERE id > 18000` Spark compares the filter against each file's
min/max and skips files that can't match, without opening them. And `count()` can be answered from
`numRecords` alone — which is why a `count()` still succeeds after `VACUUM` deletes the files an old
version needs (section 10).

In [ ]:
import json


def show_commit(version: int) -> None:
    """The actions inside one commit file."""
    path = TABLE / "_delta_log" / f"{version:020d}.json"
    print(path.name)
    for line in path.open():
        ((kind, body),) = json.loads(line).items()
        if kind == "add":
            stats = json.loads(body["stats"])
            ids = f"{stats['minValues']['id']}-{stats['maxValues']['id']}"
            print(f"  {kind:<11} {body['path'][:28]}… size={body['size']:>6} rows={stats['numRecords']:>5} ids {ids}")
        elif kind == "remove":
            print(f"  {kind:<11} {body['path'][:28]}…")
        elif kind == "commitInfo":
            print(f"  {kind:<11} {body['operation']} {body.get('operationParameters')}")
        elif kind == "metaData":
            fields = [f"{f['name']}:{f['type']}" for f in json.loads(body["schemaString"])["fields"]]
            print(f"  {kind:<11} schema {fields}")
        else:
            print(f"  {kind:<11} {body}")


show_commit(0)

### The two kinds of `.crc`

- **`_delta_log/NNN.crc` is Delta's own**, and it's JSON, not a checksum: totals, the schema and
  protocol, a histogram of file sizes, and for a small table every live file. Delta uses it to check
  the state it rebuilt from the commits, and to answer "how big is this table?" without replaying the
  log. It's a cache — delete it and the table still works.
- **`.something.crc` (leading dot) is Hadoop's**, written for every file Spark saves locally: an
  8-byte header plus one CRC32 per 512 bytes. So `.00000000000000000000.crc.crc` is the Hadoop
  checksum *of Delta's checksum file*. These don't exist on HDFS or S3.

`_delta_log/_staged_commits/` stays empty here. It's used by **coordinated commits**, where an
external coordinator such as Unity Catalog decides which commit wins: writers stage a commit there
and it is "backfilled" into `NNN.json` once accepted. A table written straight to a path commits to
`NNN.json` directly.

In [ ]:
crc = json.loads((TABLE / "_delta_log" / f"{0:020d}.crc").read_text())
print("keys:", list(crc))
print("tableSizeBytes:", crc["tableSizeBytes"], "| numFiles:", crc["numFiles"], "| protocol:", crc["protocol"])
print("live files:", [f["path"][:28] + "…" for f in crc["allFiles"]])

## 3. Append — version 1

An append writes new files and leaves the existing ones alone. The commit contains only `add`
actions, so `isBlindAppend` is true (you'll see it in section 6).

In [ ]:
new_users = spark.createDataFrame(
    [(20_000 + i, f"user_{20_000 + i:06d}") for i in range(5)], "id long, name string"
).coalesce(1)  # one small file instead of one per partition
new_users.write.format("delta").mode("append").save(str(TABLE))

print("rows:", users.toDF().count())
show_history(users)

## 4. Delete — version 2

Remove the users with `id < 100`. Parquet files can't be edited in place, so Delta:

1. finds the files that contain matching rows — only the first file, which holds ids 0–9083,
2. writes a **new** file with the rows of that file that stay,
3. commits a `remove` for the old file and an `add` for the new one.

The old file stays on disk, because older versions still need it.

In [ ]:
users.delete("id < 100")  # same as users.delete(F.col("id") < 100)

print("rows:", users.toDF().count())
show_history(users)
show_files(users)

**What to look for**

- In the history row for version 2, compare `numDeletedRows` with `numCopiedRows`: to delete 100
  rows, Delta copied every other row of that file into a new one.
- In the file list, the original 9,084-row file is still on disk, but `in current version` is
  `False`. Its replacement (8,984 rows) is the newest file.
- Rewriting whole files is the default. With deletion vectors enabled
  (`delta.enableDeletionVectors = true`), Delta instead writes a small side file marking which
  rows are deleted.

## 5. Update — version 3

Give the users with `id < 200` a `vip_` prefix. An update works like a delete: files with matching
rows are rewritten with the new values.

In [ ]:
users.update(condition=F.col("id") < 200, set={"name": F.concat(F.lit("vip_"), F.col("name"))})

users.toDF().where("id BETWEEN 195 AND 204").orderBy("id").show()
show_history(users)

## 6. The history

Each row of `history()` is one commit in `_delta_log/`, newest first:

| Column | Meaning |
|---|---|
| `version` | Commit number: `_delta_log/00000000000000000003.json` is version 3 |
| `timestamp` | When the commit happened; used by `timestampAsOf` |
| `operation` | `WRITE`, `DELETE`, `UPDATE`, `MERGE`, `RESTORE`, … |
| `operationParameters` | What was asked: write mode, delete predicate, … |
| `operationMetrics` | What happened: rows and files added, removed, copied |
| `readVersion` | The version the operation started from |
| `isBlindAppend` | `true` if the commit only added data without reading the table |

The full history has more columns. `userId`, `job`, `notebook` and `clusterId` are filled in on
Databricks and stay empty locally.

In [ ]:
history = users.history()
history.printSchema()
display_dataframe(history.select("version", "timestamp", "operation", "readVersion", "isBlindAppend").toPandas())

# Keep a copy of the history as JSON (a folder with one JSON-lines file per partition)
history.write.format("json").mode("overwrite").save("data/delta_history.json")

## 7. Time travel

Every version stays readable until its files are removed by `VACUUM`. Pick a version by number
(`versionAsOf`) or by time (`timestampAsOf`).

In [ ]:
for version in range(4):
    df = spark.read.format("delta").option("versionAsOf", version).load(str(TABLE))
    print(f"version {version}: {df.count():>6} rows")

`timestampAsOf` returns the latest version committed **at or before** the given time. Here we use
the exact commit time of version 1, taken from the history.

In [ ]:
commits = users.history().select("version", "timestamp").orderBy("version").collect()
v1_time = commits[1].timestamp
print("version 1 was committed at", v1_time)

df = spark.read.format("delta").option("timestampAsOf", str(v1_time)).load(str(TABLE))
print("rows as of that time:", df.count())

The same in SQL:

In [ ]:
spark.sql(f"SELECT COUNT(*) AS rows FROM delta.`{TABLE}` VERSION AS OF 0").show()

### What changed between two versions?

Read both versions and subtract one from the other with `exceptAll`.

In [ ]:
def version(n: int):
    return spark.read.format("delta").option("versionAsOf", n).load(str(TABLE))


print("Rows deleted by version 2 (in v1, not in v2):")
version(1).exceptAll(version(2)).orderBy("id").show(5)

print("Rows changed by version 3 (new values: in v3, not in v2):")
version(3).exceptAll(version(2)).orderBy("id").show(5)

## 8. Restore — version 4

Rolling back doesn't erase history. `restoreToVersion` writes a **new** commit that switches the
table back to the files of the old version, so versions 1–3 stay readable.

In [ ]:
users.restoreToVersion(0)

print("rows:", users.toDF().count())
show_history(users)
show_files(users)

## 9. OPTIMIZE — compact small files

Every append, delete and update leaves more, smaller files behind: this table started with 3 and now
has several, including 1-row files from the append. Lots of small files make reads slow, because each
one costs an open, a footer read and a task. `OPTIMIZE` rewrites them into fewer large ones (up to
`spark.databricks.delta.optimize.maxFileSize`, 1 GB by default).

It's a normal commit: the old files are `remove`d, the compacted ones `add`ed, `dataChange` is false
(so streaming readers ignore it), and nothing is deleted from disk. Typical use is a nightly job over
a table a stream has been appending to all day.

In [ ]:
# users is a DeltaTable object, not a DataFrame, so the maintenance commands live on it.
# Before compaction: how many files does the table have?
show_files(users)

In [ ]:
# compact small files into larger ones, which is especially useful after the restore
users.optimize().executeCompaction()

# 'in current version' is the column to watch: the old files stay on disk until VACUUM.

In [ ]:
show_files(users)

## 10. VACUUM — deleting files for real

Nothing so far has deleted a single byte. `remove` actions and `OPTIMIZE` only take files out of the
*current* version, which is exactly what makes time travel possible.

`VACUUM` is what physically deletes files that no longer belong to the current version and are older
than the retention period — **7 days by default**. Delta refuses a shorter retention unless you turn
off the safety check, because a running query or an in-flight writer may still need those files. Zero
retention is fine here only because this is a throwaway table.

The cost is history: once the files are gone, the versions that referenced them can no longer be
read.

In [ ]:
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")  # demo only
users.vacuum(retentionHours=0)  # delete every file the current version doesn't need

In [ ]:
show_files(users)

Now try to travel back to version 0. `count()` still returns 20,000, because Delta answers it from
the row counts in the log without opening a file — so **never use `count()` to check whether an old
version is still readable**. Asking for the rows themselves fails: the files are gone.

In [ ]:
# Time travel after VACUUM: count() still answers from the log stats ...
v0 = spark.read.format("delta").option("versionAsOf", 0).load(str(TABLE))
print("version 0 count():", v0.count())

# ... but reading the actual rows needs the files, which VACUUM deleted.
# Spark logs the failed task as a long Java stack trace, so quieten it for this one cell.
spark.sparkContext.setLogLevel("OFF")
try:
    v0.show(5)
except Exception as error:
    reason = next((line for line in str(error).splitlines() if "FAILED_READ_FILE" in line), str(error))
    print("reading the rows failed:", reason.strip()[:160])
finally:
    spark.sparkContext.setLogLevel("ERROR")

In [ ]:
spark.stop()